# Lab 20 extra: How to price an agent's calls and stop at a budget

## Goal

Price every model call an agent makes from the usage it reports, including input read from the prompt cache and
input written to it, and stop a run before the call that would go over a budget. We use the Chinook desk agent with
three database lookups. One question takes three model calls, and each call sends the whole conversation again, so
the input grows call by call. When the agent carries a long block of instructions, the first call writes it to the
prompt cache and later calls read it from there at a tenth of the input price.

## What you will learn

- How to read a run's usage request by request with `request_usage_entries`.
- How to price one request from the four prices in `models.env`: input, cached input, cache writes and output.
- How a long instructions block is written to the prompt cache by the first request and read by the next ones, and
  what that does to the price.
- How to stop an agent before the call that would pass a budget, with a run hook, and what such a stop cannot
  prevent.

## Setup

The first cell is the same in every lab: it finds the repository folder, reads the model name and the prices from
`models.env`, and takes the key from `.env` without printing it (lab 00 explains it line by line). The second sets a
60-second limit on every model request.

In [1]:
from pathlib import Path
import os
from dotenv import dotenv_values

# The repository folder is the first folder, from here upwards, that holds models.env
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "models.env").exists())

# Models and prices come from models.env; a non-empty value in .env replaces one of them
settings = dotenv_values(ROOT / "models.env")
local = dotenv_values(ROOT / ".env")
settings.update({name: value for name, value in local.items() if name in settings and value})

# The key comes from the environment or .env, and is never printed
os.environ.setdefault("OPENAI_API_KEY", local.get("OPENAI_API_KEY") or "")
assert os.environ["OPENAI_API_KEY"], "Put OPENAI_API_KEY in .env at the repository folder (copy .env.example)."

MODEL = settings["MODEL_DEFAULT"]
EFFORT = settings["REASONING_EFFORT"] or None
print("model:", MODEL, "| reasoning effort:", EFFORT, "| key found: yes")

model: gpt-6-luna | reasoning effort: none | key found: yes


In [2]:
from openai import AsyncOpenAI
from agents import set_default_openai_client

set_default_openai_client(AsyncOpenAI(timeout=60, max_retries=2))

---

## Part A: Price each request of one run from its usage

The desk agent here has three tools that read the shop's database (opened read-only): `find_customer`,
`get_invoices` and `get_support_rep`.

In [3]:
import sqlite3
from contextlib import closing
from agents import Agent, Runner, ModelSettings, function_tool
from openai.types.shared import Reasoning

DB_URI = f"file:{ROOT / 'data/chinook.db'}?mode=ro"


@function_tool
def find_customer(name: str) -> dict:
    """Find a customer by full name or first name; returns all matches."""
    with closing(sqlite3.connect(DB_URI, uri=True)) as conn:
        rows = conn.execute(
            "SELECT CustomerId, FirstName || ' ' || LastName FROM customers "
            "WHERE FirstName || ' ' || LastName = ? OR FirstName = ?", (name, name)).fetchall()
    return {"matches": [{"customer_id": r[0], "name": r[1]} for r in rows]}


@function_tool
def get_invoices(customer_id: int) -> dict:
    """Read this customer's invoice count and total in US dollars."""
    with closing(sqlite3.connect(DB_URI, uri=True)) as conn:
        count, total = conn.execute(
            "SELECT COUNT(*), ROUND(COALESCE(SUM(Total), 0), 2) FROM invoices WHERE CustomerId = ?",
            (customer_id,)).fetchone()
    return {"customer_id": customer_id, "invoice_count": count, "total": total}


@function_tool
def get_support_rep(customer_id: int) -> dict:
    """Read this customer's assigned support representative."""
    with closing(sqlite3.connect(DB_URI, uri=True)) as conn:
        row = conn.execute(
            "SELECT e.FirstName || ' ' || e.LastName FROM customers c "
            "JOIN employees e ON e.EmployeeId = c.SupportRepId WHERE c.CustomerId = ?", (customer_id,)).fetchone()
    return {"customer_id": customer_id, "support_rep": row[0] if row else None}

The agent gets the three tools and short instructions. Helena's question needs all three lookups.

In [4]:
INSTRUCTIONS = (
    "You answer questions for the Chinook music shop's support desk. Use the tools for shop facts. "
    "Find the customer before using their ID. Keep the reply short."
)
desk = Agent(
    name="Desk agent",
    instructions=INSTRUCTIONS,
    tools=[find_customer, get_invoices, get_support_rep],
    model=MODEL,
    model_settings=ModelSettings(reasoning=Reasoning(effort=EFFORT)),
)

QUESTION = "How many invoices does Helena Holý have, what did she spend, and who is her support rep?"
result = await Runner.run(desk, QUESTION)
print(result.final_output)

Helena Holý has **7 invoices**, totaling **$49.62**. Her support rep is **Steve Johnson**.


`result.context_wrapper.usage` adds up the tokens of every model request in the run. Its `request_usage_entries`
list keeps one entry per request, in order. Each entry has:

| Field | What it counts |
|---|---|
| `input_tokens` | every input token the request sent: instructions, tools, question, earlier replies and tool results |
| `input_tokens_details.cached_tokens` | the part of the input read from the prompt cache |
| `input_tokens_details.cache_write_tokens` | the part of the input written to the prompt cache |
| `output_tokens` | the tokens the model wrote, reasoning included |

In [5]:
entries = result.context_wrapper.usage.request_usage_entries
for number, entry in enumerate(entries, start=1):
    details = entry.input_tokens_details
    print(f"request {number}: input {entry.input_tokens} (cached {details.cached_tokens}, "
          f"written {details.cache_write_tokens}) | output {entry.output_tokens}")

request 1: input 186 (cached 0, written 0) | output 21
request 2: input 238 (cached 0, written 0) | output 52
request 3: input 349 (cached 0, written 0) | output 30


Three requests, one per turn of the agent's loop. The input grows from 186 to 238 to 349 tokens, because each request sends the whole conversation again, now with the model's tool calls and the tool results in it. Nothing was read from or written to the cache: these prompts are shorter than 1,024 tokens.

### The price of one request

`models.env` holds four prices, in US dollars per million tokens. Cached reads and cache writes are parts of
`input_tokens`, so we take them out to get the ordinary input; otherwise those tokens would be charged twice:

$$\text{dollars} = \frac{\text{ordinary} \times P_{in} + \text{cached} \times P_{cached} + \text{written} \times P_{write} + \text{output} \times P_{out}}{1{,}000{,}000}$$

where ordinary = input − cached − written. Some models report `cache_write_tokens` as `None`, so `or 0` turns a missing
count into 0.

In [6]:
PRICE = {name: float(settings[name]) for name in [
    "PRICE_INPUT_PER_MILLION", "PRICE_CACHED_INPUT_PER_MILLION",
    "PRICE_CACHE_WRITE_PER_MILLION", "PRICE_OUTPUT_PER_MILLION"]}
print(PRICE)


def dollars(entry):
    cached = entry.input_tokens_details.cached_tokens or 0
    written = entry.input_tokens_details.cache_write_tokens or 0
    ordinary = entry.input_tokens - cached - written
    return (ordinary * PRICE["PRICE_INPUT_PER_MILLION"]
            + cached * PRICE["PRICE_CACHED_INPUT_PER_MILLION"]
            + written * PRICE["PRICE_CACHE_WRITE_PER_MILLION"]
            + entry.output_tokens * PRICE["PRICE_OUTPUT_PER_MILLION"]) / 1_000_000


first = entries[0]
print(f"request 1: {first.input_tokens} input and {first.output_tokens} output tokens = ${dollars(first):.6f}")

{'PRICE_INPUT_PER_MILLION': 0.1, 'PRICE_CACHED_INPUT_PER_MILLION': 0.01, 'PRICE_CACHE_WRITE_PER_MILLION': 0.125, 'PRICE_OUTPUT_PER_MILLION': 0.5}
request 1: 186 input and 21 output tokens = $0.000029


Request 1 with its values: (186 × 0.10 + 21 × 0.50) / 1,000,000 = $0.000029. There is nothing cached or written here, so only the input and output prices apply.

### A ledger for the run

A **ledger** is one row per request, with its tokens and its price, and a total at the bottom. It shows where a run
spent its money, which a total alone hides.

In [7]:
def print_ledger(entries):
    for number, entry in enumerate(entries, start=1):
        details = entry.input_tokens_details
        print(f"request {number} | input {entry.input_tokens:5} | cached {details.cached_tokens or 0:5} | "
              f"written {details.cache_write_tokens or 0:5} | output {entry.output_tokens:4} | ${dollars(entry):.6f}")
    print(f"total: ${sum(dollars(entry) for entry in entries):.6f}")


print_ledger(entries)

request 1 | input   186 | cached     0 | written     0 | output   21 | $0.000029
request 2 | input   238 | cached     0 | written     0 | output   52 | $0.000050
request 3 | input   349 | cached     0 | written     0 | output   30 | $0.000050
total: $0.000129


The run cost $0.000129. Requests 2 and 3 cost more than request 1: they carry more input, and request 2 wrote the longest output (52 tokens, for the two tool calls). An output token costs five times an input token at these prices, so a short reply matters as much as a short prompt.

**Checkpoint A.** Three requests for Helena's question, with the input growing from one request to the next, nothing
cached or written, and a total well under a tenth of a US cent.

---

## Part B: Write a long instructions block to the prompt cache, then read it

The desk could give the agent the shop's whole policy handbook in its instructions, so it can answer any policy
question without a tool. That makes every request long. OpenAI keeps the start of a long prompt (1,024 tokens or
more) in a **prompt cache**: the first request that sends it pays to **write** it to the cache, at 1.25 times the
input price, and later requests that start the same way **read** it at a tenth of the input price.

The handbook is the 20 policy files in `data/corpus/`. `clone` copies the agent with the longer instructions.
`prompt_cache_key` groups requests that share a prompt. `uuid4` makes a new key on every run, and a new key starts from an empty cache, so the first request writes the handbook to the cache even if an earlier run of this cell already did. `extra_args` sends `prompt_cache_key`, a field the SDK has no setting for, straight to the API.

In [8]:
import uuid

CORPUS = ROOT / "data/corpus"
handbook = "\n\n".join(path.read_text() for path in sorted(CORPUS.glob("policy_*.md")))
print("policy files:", len(sorted(CORPUS.glob("policy_*.md"))), "| characters:", len(handbook))

desk_with_handbook = desk.clone(
    instructions=INSTRUCTIONS + "\n\nThe shop's policies:\n\n" + handbook,
    model_settings=ModelSettings(reasoning=Reasoning(effort=EFFORT),
                                 extra_args={"prompt_cache_key": f"lab20-{uuid.uuid4().hex[:8]}"}),
)
result = await Runner.run(desk_with_handbook, QUESTION)
print(result.final_output)
print_ledger(result.context_wrapper.usage.request_usage_entries)

policy files: 20 | characters: 6669


Helena Holý has **7 invoices**, totaling **$49.62**. Her support rep is **Steve Johnson**.
request 1 | input  1595 | cached     0 | written  1592 | output   37 | $0.000218
request 2 | input  1647 | cached  1592 | written    52 | output   52 | $0.000049
request 3 | input  1758 | cached  1644 | written   111 | output   30 | $0.000046
total: $0.000312


Request 1 wrote 1,592 tokens to the cache (the instructions with the handbook, and the tool descriptions) and cost $0.000210, seven times the plain agent's first request. Requests 2 and 3 read 1,592 and 1,644 tokens from the cache and cost $0.000049 and $0.000046, no more than the plain agent's. Each request also writes its new end (52, then 111 tokens), so the next request can read it. The whole run cost $0.000304, against $0.000129 for the plain agent.

A second question with the same agent starts with the same handbook, so its first request can read the cache too:

In [9]:
result = await Runner.run(desk_with_handbook, "Who is Puja Srivastava's support rep?")
print(result.final_output)
print_ledger(result.context_wrapper.usage.request_usage_entries)

Puja Srivastava’s support representative is Jane Peacock.
request 1 | input  1586 | cached  1567 | written    16 | output   23 | $0.000029
request 2 | input  1642 | cached  1583 | written    56 | output   20 | $0.000033
request 3 | input  1689 | cached  1639 | written    47 | output   17 | $0.000031
total: $0.000094


This time request 1 read 1,567 tokens from the cache and wrote only 16: the handbook was already there. The whole run cost $0.000094. The question and the replies differ from Helena's, so the two totals cannot be compared directly, but this run shows that once the handbook is in the cache, its 1,500
tokens add little to a request's price.

**In production:** by default a cache entry is kept for at least 30 minutes (its minimum lifetime), and each read keeps it for at least 30 minutes after that read ([OpenAI: cache lifetime](https://developers.openai.com/api/docs/guides/prompt-caching#cache-lifetime)). Put what stays the same (instructions, tool descriptions, a handbook) at the start of the prompt and what
changes (the question) at the end, or every request writes again.

**Checkpoint B.** The handbook run's first request writes about as many tokens as the handbook and instructions hold
(ours: 1,592); the next requests read them. The second question's first request reads the cache.

---

## Part C: Stop a run before the call that would pass a budget

A budget is a limit on what one run may spend. The Agents SDK calls a **run hook** at fixed points of a run; a class
based on `RunHooks` overrides the points it needs. Ours uses `on_llm_start`, which the SDK calls just before each
model request. It prices the requests made so far (`context.usage.request_usage_entries`), and if they already reach
the budget it raises `BudgetReached`, which stops the run.

In [10]:
from agents import RunHooks


class BudgetReached(Exception):
    pass


class Budget(RunHooks):
    def __init__(self, limit_usd):
        self.limit_usd = limit_usd

    async def on_llm_start(self, context, agent, system_prompt, input_items):
        spent = sum(dollars(entry) for entry in context.usage.request_usage_entries)
        if spent >= self.limit_usd:
            raise BudgetReached(f"spent ${spent:.6f} of ${self.limit_usd:.6f}; no further model calls")

We give the plain desk agent a budget smaller than one request's price (a hundred-thousandth of a dollar). The first
request goes out, because nothing has been spent yet; before the second, the hook finds the budget reached.

In [11]:
try:
    result = await Runner.run(desk, QUESTION, hooks=Budget(limit_usd=0.00001))
    print(result.final_output)
except BudgetReached as stop:
    print("stopped:", stop)

stopped: spent $0.000029 of $0.000010; no further model calls


Stopped after one request. The first request cost $0.000029, already more than the $0.00001 budget, so the hook raised `BudgetReached` before the second, and the run ended with no answer. Notice that the first request alone went over the budget: a hook that runs before a call can only stop the next one.

With a budget of one US cent, the same question runs to its answer:

In [12]:
result = await Runner.run(desk, QUESTION, hooks=Budget(limit_usd=0.01))
print(result.final_output)
print_ledger(result.context_wrapper.usage.request_usage_entries)

Helena Holý has **7 invoices**, spending **$49.62** in total. Her support rep is **Steve Johnson**.
request 1 | input   186 | cached     0 | written     0 | output   21 | $0.000029
request 2 | input   238 | cached     0 | written     0 | output   52 | $0.000050
request 3 | input   349 | cached     0 | written     0 | output   32 | $0.000051
total: $0.000130


**What this budget cannot prevent:** it stops between calls. The call that crosses the limit has already been sent, so
a run can end a little over its budget. It also counts only the requests that came back: a request that failed and
was retried can still be billed. In production, set a spending limit on the account too.

**Checkpoint C.** With the tiny budget: `stopped:` and the amount spent by one request. With one US cent: Helena's
three facts and a ledger like Part A's.

---

## What you can do now

- Read a run's usage request by request, cached and written input included (Part A).
- Price a request from the four prices in `models.env`, and keep a ledger for a run (Part A).
- Explain what a prompt cache write and a cache read cost, and keep the part of a prompt that stays the same at its
  start (Part B).
- Stop an agent before the call that would pass a budget, and say what such a stop cannot prevent (Part C).

## Troubleshooting

| What you see | Why | Fix |
|---|---|---|
| `TypeError: unsupported operand type(s) for *: 'NoneType' and 'float'` | the model reports a token count as `None` | price with `or 0`, as `dollars` does |
| `cached 0` and `written 0` on every handbook request | the instructions block is shorter than 1,024 tokens, or the model has no prompt cache | check the request's input tokens; caching starts at 1,024 |

## Resources

- [OpenAI: prompt caching](https://developers.openai.com/api/docs/guides/prompt-caching): what is cached, for how
  long, and `prompt_cache_key`
- [OpenAI: pricing](https://developers.openai.com/api/docs/pricing): input, cached input and output prices per model
- [OpenAI Agents SDK: usage](https://openai.github.io/openai-agents-python/usage/): `context_wrapper.usage` and
  per-request entries
- [OpenAI Agents SDK: lifecycle hooks](https://openai.github.io/openai-agents-python/ref/lifecycle/): `RunHooks` and
  `on_llm_start`

Back to [notebook 2](20b_evals_and_tracing_at_the_desk.ipynb) or [the lab's README](README.md).